**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 27: DPO 학습 실습 (SFT + DPO)

## 🎯 DPO 학습 실습 개요

이 노트북에서는 **Qwen2.5-1.5B-Instruct** 모델을 사용하여 SFT → DPO 파이프라인을 실습합니다.  
RTX 4060(8GB VRAM) 환경에 최적화된 설정으로 진행합니다.

### 학습 목표
- 🎯 SFT → DPO 2단계 파이프라인 구현
- 1️⃣ 4bit 양자화 + LoRA로 메모리 효율적인 DPO 학습
- 2️⃣ DPO 핵심 파라미터(beta, loss_type) 이해
- 3️⃣ SFT Only vs SFT+DPO 성능 비교

### GPU 요구사항
- ✅ GPU 필수 (RTX 4060 8GB 이상)
- 모델: Qwen2.5-1.5B-Instruct (4bit 양자화)
- 예상 VRAM: ~5-6GB

### 파이프라인 개요
```
Step 1: SFT 학습 → 기본 능력 향상
Step 2: DPO 학습 → 선호도 정렬
```

---

In [1]:
# 필요한 라이브러리 임포트
import torch
import gc
import json
import os
from pathlib import Path
from datetime import datetime

# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print("✅ 기본 라이브러리 임포트 완료")
print(f"📅 실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
else:
    print("❌ GPU를 사용할 수 없습니다. 이 노트북은 GPU가 필수입니다.")

✅ 기본 라이브러리 임포트 완료
📅 실행 시각: 2026-04-06 11:15:13
🔥 PyTorch: 2.10.0+cu128
🔥 CUDA: 12.8
✅ GPU: NVIDIA TITAN RTX
[초기 상태] GPU: 0.0GB / 23.5GB


In [2]:
# 추가 라이브러리 임포트
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import Dataset

print("✅ 모든 라이브러리 임포트 완료")
print("   • transformers, peft, trl, datasets")

/home/ejkim/multicampus/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ejkim/multicampus/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


✅ 모든 라이브러리 임포트 완료
   • transformers, peft, trl, datasets


---
## 1️⃣ 환경 설정 및 모델 로드 (Qwen2.5-1.5B-Instruct, 4bit)

### 모델 선택 이유
- Qwen2.5-1.5B-Instruct: 한국어 성능이 좋은 소형 모델
- 4bit 양자화: RTX 4060(8GB) 환경에서 DPO 학습 가능
- LoRA: 학습 파라미터를 최소화하여 메모리 절약

In [3]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("./outputs/dpo_training")
SFT_OUTPUT = OUTPUT_DIR / "sft_checkpoint"
DPO_OUTPUT = OUTPUT_DIR / "dpo_checkpoint"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 60)
print("📌 모델 설정")
print("=" * 60)
print(f"   모델: {MODEL_NAME}")
print(f"   양자화: 4bit NF4 (double quantization)")
print(f"   SFT 출력: {SFT_OUTPUT}")
print(f"   DPO 출력: {DPO_OUTPUT}")

📌 모델 설정
   모델: Qwen/Qwen2.5-1.5B-Instruct
   양자화: 4bit NF4 (double quantization)
   SFT 출력: outputs/dpo_training/sft_checkpoint
   DPO 출력: outputs/dpo_training/dpo_checkpoint


In [4]:
# 모델 및 토크나이저 로드
print("=" * 60)
print("📌 모델 로딩")
print("=" * 60)

print("🔄 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # DPO에서는 left padding 권장
print(f"   ✅ 토크나이저 로드 완료 (vocab: {tokenizer.vocab_size})")

print("🔄 모델 로딩...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.enable_input_require_grads()
model.config.use_cache = False  # 학습 시 캐시 비활성화

print(f"   ✅ 모델 로드 완료")
print_gpu_memory("모델 로드 후")

📌 모델 로딩
🔄 토크나이저 로딩...
   ✅ 토크나이저 로드 완료 (vocab: 151643)
🔄 모델 로딩...


`torch_dtype` is deprecated! Use `dtype` instead!


   ✅ 모델 로드 완료
[모델 로드 후] GPU: 1.1GB / 23.5GB


In [5]:
# LoRA 설정
print("=" * 60)
print("📌 LoRA 설정")
print("=" * 60)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"   LoRA rank (r): {lora_config.r}")
print(f"   LoRA alpha: {lora_config.lora_alpha}")
print(f"   Target modules: {lora_config.target_modules}")
print(f"   Dropout: {lora_config.lora_dropout}")

# 학습 가능 파라미터 확인
peft_model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"\n📊 파라미터 현황:")
print(f"   학습 가능: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"   전체: {total:,}")
print_gpu_memory("LoRA 적용 후")

# peft_model은 SFT에서 사용할 것이므로 여기서 해제
del peft_model
gc.collect()
torch.cuda.empty_cache()

📌 LoRA 설정
   LoRA rank (r): 16
   LoRA alpha: 32
   Target modules: {'k_proj', 'q_proj', 'o_proj', 'v_proj'}
   Dropout: 0.05

📊 파라미터 현황:
   학습 가능: 4,358,144 (0.49%)
   전체: 892,974,592
[LoRA 적용 후] GPU: 1.1GB / 23.5GB


---
## 2️⃣ Step 1: SFT 학습 (기본 능력 향상)

DPO 학습 전에 먼저 SFT로 모델의 기본 능력을 향상시킵니다.

### SFT 데이터
- 한국어 instruction-response 쌍
- Session 26에서 만든 preference 데이터의 chosen 응답을 활용

In [6]:
# SFT 학습 데이터 준비
print("=" * 60)
print("📌 SFT 학습 데이터 준비")
print("=" * 60)

# 샘플 SFT 데이터 (preference 데이터의 chosen을 활용)
sft_data = [
    {
        "instruction": "파이썬에서 리스트와 튜플의 차이점을 설명해주세요.",
        "response": (
            "파이썬에서 리스트(list)와 튜플(tuple)은 모두 순서가 있는 컬렉션이지만, "
            "중요한 차이점이 있습니다.\n\n"
            "1. 변경 가능성: 리스트는 변경 가능(mutable), 튜플은 변경 불가(immutable)\n"
            "2. 문법: 리스트는 [1,2,3], 튜플은 (1,2,3)\n"
            "3. 성능: 튜플이 약간 더 빠르고 메모리를 적게 사용\n"
            "4. 사용 시점: 변경이 필요하면 리스트, 고정 데이터면 튜플"
        )
    },
    {
        "instruction": "좋은 코드 리뷰를 하려면 어떻게 해야 하나요?",
        "response": (
            "좋은 코드 리뷰를 위한 핵심 원칙:\n\n"
            "1. 마인드셋: 코드를 비판하되, 사람을 비판하지 않기\n"
            "2. 확인 항목: 로직 정확성, 가독성, 테스트 커버리지, 보안\n"
            "3. 실천 팁: PR 크기 작게 유지, 24시간 이내 리뷰, 긍정적 피드백도 함께\n"
            "4. 제안 방식: '이것은 틀렸다' 대신 '이렇게 하면 어떨까요?'"
        )
    },
    {
        "instruction": "머신러닝 모델의 과적합을 방지하는 방법은?",
        "response": (
            "과적합(Overfitting) 방지 방법:\n\n"
            "1. 데이터 측면: 더 많은 데이터 수집, 데이터 증강\n"
            "2. 모델 측면: 정규화(L1/L2), 드롭아웃, 모델 복잡도 줄이기\n"
            "3. 학습 과정: 조기 종료(Early Stopping), 교차 검증\n"
            "4. 가장 효과적인 방법은 양질의 데이터를 충분히 확보하는 것입니다."
        )
    },
    {
        "instruction": "REST API와 GraphQL의 차이점은 무엇인가요?",
        "response": (
            "REST API와 GraphQL 비교:\n\n"
            "REST API: 리소스 기반 URL, HTTP 메서드 활용, 고정된 응답 구조\n"
            "GraphQL: 단일 엔드포인트, 클라이언트가 필요한 데이터만 요청, 강력한 타입 시스템\n\n"
            "선택 기준: 간단한 CRUD는 REST, 복잡한 데이터 관계는 GraphQL,"
            "캐싱 중요시 REST, 모바일 앱은 GraphQL이 유리합니다."
        )
    },
    {
        "instruction": "도커(Docker)를 사용하는 이유와 장점은?",
        "response": (
            "Docker의 주요 장점:\n\n"
            "1. 환경 일관성: '내 PC에서는 되는데...' 문제 해결\n"
            "2. 빠른 배포: 이미지 기반으로 몇 초만에 환경 구축\n"
            "3. 격리성: 프로세스, 네트워크, 파일시스템 격리\n"
            "4. 확장성: K8s와 결합하여 쉬운 스케일링\n"
            "5. 리소스 효율: VM 대비 가벼움 (OS 커널 공유)"
        )
    },
    {
        "instruction": "효과적인 프롬프트 엔지니어링 기법은?",
        "response": (
            "프롬프트 엔지니어링 핵심 기법:\n\n"
            "기본: 역할 부여, 구체적 지시, 출력 형식 지정\n"
            "고급: Few-shot (예시 2-3개), Chain-of-Thought, Self-Consistency\n"
            "실전 팁: 구분자로 영역 구분, 부정문보다 긍정문, 반복 개선"
        )
    },
    {
        "instruction": "데이터베이스 인덱스가 왜 중요한가요?",
        "response": (
            "데이터베이스 인덱스의 중요성:\n\n"
            "원리: 책의 색인처럼 데이터 위치를 빠르게 찾음\n"
            "성능: 100만 건에서 Full Scan → O(n) 수초, Index → O(log n) 수ms\n"
            "주의: INSERT/UPDATE 시 유지 비용, 과도한 인덱스는 오히려 성능 저하\n"
            "기준: WHERE, JOIN, ORDER BY에 사용되는 컬럼에 생성"
        )
    },
    {
        "instruction": "파이썬에서 비동기 프로그래밍의 장점은?",
        "response": (
            "파이썬 비동기(async/await) 프로그래밍:\n\n"
            "장점: 높은 동시성, 리소스 효율, 수천 동시 연결 가능\n"
            "적합: 웹 서버(FastAPI), API 호출, 파일/DB I/O\n"
            "부적합: CPU 바운드 작업, 간단한 스크립트\n"
            "I/O 대기 동안 다른 작업을 처리할 수 있어 효율적입니다."
        )
    },
]

# 채팅 형식으로 변환
def format_sft_data(data_list):
    formatted = []
    for item in data_list:
        messages = [
            {"role": "system", "content": "당신은 친절하고 도움이 되는 AI 어시스턴트입니다."},
            {"role": "user", "content": item["instruction"]},
            {"role": "assistant", "content": item["response"]}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({"text": text})
    return formatted

sft_formatted = format_sft_data(sft_data)
sft_dataset = Dataset.from_list(sft_formatted)

print(f"✅ SFT 데이터 준비 완료: {len(sft_dataset)}개 샘플")
print(f"\n📋 첫 번째 샘플 미리보기:")
print(sft_dataset[0]["text"][:300] + "...")

📌 SFT 학습 데이터 준비
✅ SFT 데이터 준비 완료: 8개 샘플

📋 첫 번째 샘플 미리보기:
<|im_start|>system
당신은 친절하고 도움이 되는 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
파이썬에서 리스트와 튜플의 차이점을 설명해주세요.<|im_end|>
<|im_start|>assistant
파이썬에서 리스트(list)와 튜플(tuple)은 모두 순서가 있는 컬렉션이지만, 중요한 차이점이 있습니다.

1. 변경 가능성: 리스트는 변경 가능(mutable), 튜플은 변경 불가(immutable)
2. 문법: 리스트는 [1,2,3], 튜플은 (1,2,3)
3. 성능: 튜플이 약간 더 빠...


In [7]:
# SFT 학습 실행
print("=" * 60)
print("📌 Step 1: SFT 학습 시작")
print("=" * 60)
print_gpu_memory("SFT 학습 전")

# SFT 학습 설정 (RTX 4060 최적화)
sft_config = SFTConfig(
    output_dir=str(SFT_OUTPUT),
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    max_length=1024,
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
)

# SFT Trainer 생성
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("\n🔥 SFT 학습 시작...")
sft_result = sft_trainer.train()

print(f"\n✅ SFT 학습 완료!")
print(f"   Training Loss: {sft_result.training_loss:.4f}")
print(f"   Steps: {sft_result.global_step}")
print_gpu_memory("SFT 학습 후")

/home/ejkim/multicampus/venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/ejkim/multicampus/venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


📌 Step 1: SFT 학습 시작
[SFT 학습 전] GPU: 1.1GB / 23.5GB


Truncating train dataset: 100%|██████████| 8/8 [00:00<00:00, 2973.37 examples/s]
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



🔥 SFT 학습 시작...


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,2.419400
2,2.419400
3,2.303900
4,2.162300
5,2.042900
6,1.958200
7,1.904500
8,1.870900
9,1.850300
10,1.841800


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn


✅ SFT 학습 완료!
   Training Loss: 2.0774
   Steps: 10
[SFT 학습 후] GPU: 1.1GB / 23.5GB


In [ ]:
# SFT 모델 저장
print("🔄 SFT 모델 저장 중...")
sft_trainer.save_model(str(SFT_OUTPUT))
tokenizer.save_pretrained(str(SFT_OUTPUT))
print(f"✅ SFT 모델 저장 완료: {SFT_OUTPUT}")

# SFT 학습 후 테스트
print("=" * 60)
print("📌 SFT 모델 테스트")
print("=" * 60)

sft_trainer.model.config.use_cache = True
sft_trainer.model.eval()
for name, param in sft_trainer.model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)

test_prompts = [
    "좋은 API 설계 원칙은 무엇인가요?",
    "깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "당신은 친절하고 도움이 되는 AI 어시스턴트입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(sft_trainer.model.device)
    
    with torch.no_grad():
        outputs = sft_trainer.model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n❓ {prompt}")
    print(f"🤖 SFT 응답: {response[:300]}")
    print("-" * 50)

🔄 SFT 모델 저장 중...
✅ SFT 모델 저장 완료: outputs/dpo_training/sft_checkpoint

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
📌 SFT 모델 테스트

❓ 좋은 API 설계 원칙은 무엇인가요?
🤖 SFT 응답: API 설계에서 중요한 원칙들로는 다음과 같습니다:

1. 명확한 기능: 각 API 호출에 대해 명확하게 설명된 함수와 결과를 제공합니다.

2. 간단성: API 구조와 호출 방식이 간단해야 합니다.

3. 확장성: API의 성능과 응답속도를 향상시키기 위해 설계되었습니다.

4. 품질 유지: API의 데이터 형식, 인코딩/디코딩 등의 품질을 유지하기 위해 고려되었습니다.

5. 보안: API의 데이터 접근 권한 및 통합 시스템 안전성을 고려했습니다.

6. 문서화: 모든 API 호출에 대한 설명과 사용 방법을 포함시킵니다.

7.
--------------------------------------------------

❓ 깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?
🤖 SFT 응답: Git 브랜치 전략 선택 시 고려해야 할 몇 가지 중요한 요소가 있습니다:

1. **개선 및 수정의 필요성**: 개선이나 수정이 필요한 경우, 별도의 브랜치를 사용하면 훨씬 쉽습니다.

2. **팀원간 협업**: 각자 개발하는 코드가 서로 독립적일수록, 별도의 브랜치를 사용하는 것이 좋습니다.

3. **리포지토리 크기와 수명**: 빠르게 변화하는 리포지토리를 유지하기 위해 별도의 브랜치를 사용하세요.
   
4. **팀원의 개입 여부**: 팀원들이 코드를 변경하거나 업데이트하는 과정에서 별도의 브랜치를 사용하는 것은 안전합니다.
--------------------------------------------------


In [9]:
# GPU 메모리 정리 (DPO 학습 준비)
print("🧹 SFT 학습 메모리 정리...")
print_gpu_memory("정리 전")

del sft_trainer
del model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료 → DPO 학습 준비")

🧹 SFT 학습 메모리 정리...
[정리 전] GPU: 1.1GB / 23.5GB
[정리 후] GPU: 0.5GB / 23.5GB
✅ 메모리 정리 완료 → DPO 학습 준비


---
## 3️⃣ Preference 데이터 로드 및 포맷팅

Session 27에서 만든 선호도 데이터를 DPO 학습 형식으로 로드합니다.

In [10]:
# Preference 데이터 로드
print("=" * 60)
print("📌 Preference 데이터 로드")
print("=" * 60)

# 샘플 데이터 경로 확인
sample_path = Path("../data/samples/preference_sample.json")

if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        raw_preference_data = json.load(f)
    print(f"✅ 파일에서 로드: {sample_path}")
else:
    print("⚠️ 샘플 파일을 찾을 수 없습니다. 내장 데이터를 사용합니다.")
    raw_preference_data = [
        {
            "prompt": "파이썬에서 리스트와 튜플의 차이점을 설명해주세요.",
            "chosen": "파이썬에서 리스트(list)와 튜플(tuple)은 모두 순서가 있는 컬렉션이지만, 중요한 차이점이 있습니다.\n\n1. 변경 가능성: 리스트는 mutable, 튜플은 immutable\n2. 문법: 리스트 [1,2,3], 튜플 (1,2,3)\n3. 성능: 튜플이 약간 더 빠름\n4. 사용: 변경 필요시 리스트, 고정 데이터는 튜플",
            "rejected": "리스트는 대괄호고 튜플은 소괄호입니다."
        },
        {
            "prompt": "좋은 코드 리뷰를 하려면 어떻게 해야 하나요?",
            "chosen": "좋은 코드 리뷰를 위한 핵심 원칙:\n\n1. 코드를 비판하되 사람은 비판하지 않기\n2. 로직 정확성, 가독성, 테스트 커버리지 확인\n3. PR 크기를 작게 유지하고 24시간 이내 리뷰\n4. '틀렸다' 대신 '이렇게 하면 어떨까요?'로 제안",
            "rejected": "코드를 잘 읽고 문제 있으면 말하면 됩니다."
        },
        {
            "prompt": "머신러닝 모델의 과적합을 방지하는 방법은?",
            "chosen": "과적합 방지 방법:\n\n1. 데이터 측면: 더 많은 데이터 수집, 데이터 증강\n2. 모델 측면: L1/L2 정규화, 드롭아웃, 모델 복잡도 줄이기\n3. 학습 과정: 조기 종료, 교차 검증, 학습률 스케줄링\n\n가장 효과적인 방법은 양질의 데이터를 충분히 확보하는 것입니다.",
            "rejected": "드롭아웃 쓰면 됩니다."
        },
        {
            "prompt": "도커(Docker)를 사용하는 이유와 장점은?",
            "chosen": "Docker의 주요 장점:\n\n1. 환경 일관성: '내 PC에서는 되는데...' 문제 해결\n2. 빠른 배포: 이미지 기반 환경 구축\n3. 격리성: 프로세스, 네트워크, 파일시스템 격리\n4. 확장성: K8s와 결합 가능\n5. 리소스 효율: VM 대비 가벼움",
            "rejected": "환경 설정 귀찮을 때 씁니다."
        },
        {
            "prompt": "효과적인 프롬프트 엔지니어링 기법은?",
            "chosen": "프롬프트 엔지니어링 핵심 기법:\n\n기본: 역할 부여, 구체적 지시, 출력 형식 지정\n고급: Few-shot, Chain-of-Thought, Self-Consistency\n실전: 구분자 활용, 긍정문 사용, 반복 개선",
            "rejected": "잘 물어보면 됩니다."
        },
    ]

print(f"\n📊 로드된 데이터: {len(raw_preference_data)}개 샘플")

📌 Preference 데이터 로드
✅ 파일에서 로드: ../data/samples/preference_sample.json

📊 로드된 데이터: 10개 샘플


In [11]:
# DPO 형식으로 변환
print("=" * 60)
print("📌 DPO 학습 형식으로 데이터 변환")
print("=" * 60)

system_prompt = "당신은 친절하고 도움이 되는 AI 어시스턴트입니다."

def format_for_dpo(data_list):
    """Preference 데이터를 DPO 형식으로 변환"""
    formatted = []
    for item in data_list:
        formatted.append({
            "chosen": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["chosen"]}
            ],
            "rejected": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["rejected"]}
            ]
        })
    return formatted

dpo_data = format_for_dpo(raw_preference_data)
dpo_dataset = Dataset.from_list(dpo_data)

print(f"✅ DPO 데이터셋 생성 완료: {len(dpo_dataset)}개 샘플")
print(f"📋 컬럼: {dpo_dataset.column_names}")
print(f"\n📋 첫 번째 샘플:")
print(f"   Chosen: {dpo_dataset[0]['chosen'][-1]['content'][:100]}...")
print(f"   Rejected: {dpo_dataset[0]['rejected'][-1]['content'][:100]}...")

📌 DPO 학습 형식으로 데이터 변환
✅ DPO 데이터셋 생성 완료: 10개 샘플
📋 컬럼: ['chosen', 'rejected']

📋 첫 번째 샘플:
   Chosen: 파이썬에서 리스트(list)와 튜플(tuple)은 모두 순서가 있는 컬렉션이지만, 중요한 차이점이 있습니다.

1. **변경 가능성(Mutability)**
   - 리스트: 변경...
   Rejected: 리스트는 대괄호고 튜플은 소괄호입니다. 리스트는 바꿀 수 있고 튜플은 못 바꿉니다....


---
## 4️⃣ Step 2: DPO 학습 (DPOTrainer from trl)

SFT 체크포인트 위에 DPO 학습을 진행합니다.

### DPO 학습 흐름
```
1. SFT 체크포인트 로드 (reference model + policy model)
2. Preference 데이터로 DPO 손실 계산
3. Policy model만 업데이트 (reference는 고정)
4. chosen 확률 ↑, rejected 확률 ↓
```

In [12]:
# DPO용 모델 다시 로드
print("=" * 60)
print("📌 DPO 학습을 위한 모델 재로드")
print("=" * 60)

# 기본 모델 로드
print("🔄 Base 모델 로딩...")
dpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
dpo_model.enable_input_require_grads()
dpo_model.config.use_cache = False

# SFT LoRA 가중치 로드
if SFT_OUTPUT.exists():
    print("🔄 SFT LoRA 가중치 로드...")
    dpo_model = PeftModel.from_pretrained(
        dpo_model,
        str(SFT_OUTPUT),
        is_trainable=True,
    )
    print("   ✅ SFT 가중치 로드 완료")
else:
    print("⚠️ SFT 체크포인트를 찾을 수 없어 새로운 LoRA를 적용합니다.")
    dpo_model = get_peft_model(dpo_model, lora_config)

print_gpu_memory("DPO 모델 로드 후")

📌 DPO 학습을 위한 모델 재로드
🔄 Base 모델 로딩...
🔄 SFT LoRA 가중치 로드...
   ✅ SFT 가중치 로드 완료
[DPO 모델 로드 후] GPU: 1.5GB / 23.5GB


In [13]:
# DPO 학습 설정 및 실행
print("=" * 60)
print("📌 DPO 학습 시작")
print("=" * 60)

# DPO 설정 (RTX 4060 최적화)
dpo_config = DPOConfig(
    output_dir=str(DPO_OUTPUT),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    fp16=True,
    max_length=1024,
    max_prompt_length=512,
    beta=0.1,                  # DPO 핵심 파라미터
    loss_type="sigmoid",       # DPO 손실 타입
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

print("📋 DPO 학습 설정:")
print(f"   beta: {dpo_config.beta}")
print(f"   loss_type: {dpo_config.loss_type}")
print(f"   batch_size: {dpo_config.per_device_train_batch_size}")
print(f"   gradient_accumulation: {dpo_config.gradient_accumulation_steps}")
print(f"   learning_rate: {dpo_config.learning_rate}")
print(f"   max_length: {dpo_config.max_length}")
print(f"   epochs: {dpo_config.num_train_epochs}")

# DPO Trainer 생성
dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("\n🔥 DPO 학습 시작...")
print_gpu_memory("DPO 학습 시작")

dpo_result = dpo_trainer.train()

print(f"\n✅ DPO 학습 완료!")
print(f"   Training Loss: {dpo_result.training_loss:.4f}")
print(f"   Steps: {dpo_result.global_step}")
print_gpu_memory("DPO 학습 후")

📌 DPO 학습 시작
📋 DPO 학습 설정:
   beta: 0.1
   loss_type: sigmoid
   batch_size: 1
   gradient_accumulation: 8
   learning_rate: 5e-05
   max_length: 1024
   epochs: 3


Tokenizing train dataset: 100%|██████████| 10/10 [00:00<00:00, 703.32 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



🔥 DPO 학습 시작...
[DPO 학습 시작] GPU: 1.5GB / 23.5GB


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
1,0.400600
2,0.458400
3,0.312400
4,0.197200
5,0.176300
6,0.134600


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/ejkim/multicampus/venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



✅ DPO 학습 완료!
   Training Loss: 0.2799
   Steps: 6
[DPO 학습 후] GPU: 1.6GB / 23.5GB


In [14]:
# DPO 모델 저장
print("🔄 DPO 모델 저장 중...")
dpo_trainer.save_model(str(DPO_OUTPUT))
tokenizer.save_pretrained(str(DPO_OUTPUT))
print(f"✅ DPO 모델 저장 완료: {DPO_OUTPUT}")

🔄 DPO 모델 저장 중...
✅ DPO 모델 저장 완료: outputs/dpo_training/dpo_checkpoint


---
## 5️⃣ DPO 학습 파라미터 설명 (beta, loss_type)

### 핵심 파라미터

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| `beta` | 0.1 | KL 페널티 강도. 높을수록 보수적 학습 |
| `loss_type` | sigmoid | DPO 손실 함수 종류 |
| `max_length` | 1024 | chosen/rejected 최대 토큰 수 |
| `max_prompt_length` | 512 | 프롬프트 최대 토큰 수 |

### beta 값에 따른 효과

```
beta = 0.01 → 매우 공격적 학습 (큰 변화, 불안정할 수 있음)
beta = 0.1  → 일반적 설정 (균형잡힌 학습)
beta = 0.5  → 매우 보수적 학습 (작은 변화, 안정적)
```

### loss_type 종류

| 타입 | 설명 |
|------|------|
| `sigmoid` | 기본 DPO 손실 (가장 일반적) |
| `hinge` | SVM 스타일 힌지 손실 |
| `ipo` | Identity Preference Optimization |
| `kto_pair` | Kahneman-Tversky Optimization |

In [15]:
# DPO 파라미터 영향 분석
print("=" * 60)
print("📌 DPO 핵심 파라미터 분석")
print("=" * 60)

import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# beta 값에 따른 그래디언트 크기 비교
log_ratio_diff = np.linspace(-3, 3, 100)

print("\n📊 beta 값에 따른 학습 강도:")
print("-" * 50)

for beta in [0.01, 0.1, 0.5]:
    # DPO 그래디언트 크기 (대략적)
    grad_magnitude = np.mean(np.abs(beta * (1 - sigmoid(beta * log_ratio_diff))))
    
    label = ""
    if beta == 0.01:
        label = "(공격적 - 큰 변화)"
    elif beta == 0.1:
        label = "(균형 - 권장값)"
    else:
        label = "(보수적 - 작은 변화)"
    
    bar = "█" * int(grad_magnitude * 100)
    print(f"   beta={beta:>4}: grad~{grad_magnitude:.4f} {bar} {label}")

print("\n💡 RTX 4060 환경에서는 beta=0.1이 안정적인 기본값입니다.")
print("   데이터가 적을 때는 beta를 높이고, 많을 때는 낮출 수 있습니다.")

📌 DPO 핵심 파라미터 분석

📊 beta 값에 따른 학습 강도:
--------------------------------------------------
   beta=0.01: grad~0.0050  (공격적 - 큰 변화)
   beta= 0.1: grad~0.0500 █████ (균형 - 권장값)
   beta= 0.5: grad~0.2500 █████████████████████████ (보수적 - 작은 변화)

💡 RTX 4060 환경에서는 beta=0.1이 안정적인 기본값입니다.
   데이터가 적을 때는 beta를 높이고, 많을 때는 낮출 수 있습니다.


---
## 6️⃣ 학습 전후 비교 (SFT Only vs SFT+DPO)

SFT만 적용한 모델과 SFT+DPO를 적용한 모델의 응답을 비교합니다.

In [16]:
# SFT vs DPO 모델 비교 테스트
print("=" * 60)
print("📌 SFT Only vs SFT+DPO 비교")
print("=" * 60)

dpo_trainer.model.config.use_cache = True
dpo_trainer.model.eval()
for name, param in dpo_trainer.model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)

def generate_response(model, prompt, max_new_tokens=256):
    """모델로 응답 생성"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# 비교 테스트 프롬프트
comparison_prompts = [
    "깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?",
    "좋은 API 설계 원칙은 무엇인가요?",
    "파이썬에서 비동기 프로그래밍의 장점은?",
]

print("\n🔥 DPO 모델 응답 생성 중...\n")

for prompt in comparison_prompts:
    print(f"{'='*50}")
    print(f"❓ 질문: {prompt}")
    print(f"{'='*50}")
    
    # DPO 모델 응답
    dpo_response = generate_response(dpo_trainer.model, prompt)
    print(f"\n🟢 SFT+DPO 응답:")
    print(f"   {dpo_response[:400]}")
    print()

📌 SFT Only vs SFT+DPO 비교

🔥 DPO 모델 응답 생성 중...

❓ 질문: 깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?

🟢 SFT+DPO 응답:
   Git의 브랜치 전략에 대한 고려사항:

1. 개발 흔적 남기기: 브랜치를 통해 각자 작업을 남기고, 이후 병합할 때 혼동되지 않도록.

2. 커밋 주기: 개별 브랜치마다 커밋 주기를 설정하여 관리하기.

3. 대형 커밋: 빠른 시간 내에 커밋을 많이 하는 경우, 라인업된 커밋으로 처리하기.

4. 코드 리뷰: 각 브랜치가 완료되면, 다른 개발자가 코드 리뷰 받도록.

5. 트래킹 브랜치: 빠른 업데이트 가능하도록 트래킹 브랜치 사용.

6. 커뮤니티: 공통 브랜치로 유지, 개발자 간 커뮤니티 활성화.

7. 배포 브랜치: 개발 후 배포 준비 브랜치를 따로 만들어서 배포 시 필요.

8. 복잡한 변경 사항: 복잡한 변경

❓ 질문: 좋은 API 설계 원칙은 무엇인가요?

🟢 SFT+DPO 응답:
   API (Application Programming Interface) 설계에는 몇 가지 중요한 원칙들이 있습니다:

1. 명확한 기능: 각 함수 또는 메서드가 완벽하게 특정 작업을 수행하도록 명시되어 있어야 합니다.

2. 응용 프로그램 간의 통합성: API는 다른 애플리케이션간에 통합성을 제공해야 합니다.

3. 확장성: API는 쉽게 확장될 수 있어야 합니다.
   
4. 보안: 데이터를 안전하게 전송하고 저장하는 방법이 필요합니다.

5. 문서화: API 사용자가 이해하기 쉬운 문서로 작성되어야 합니다.

6. 테스트 가능한: 모든 API 함수가 동작할 때마다 테스트가 이루어져야 합니다.

7. 개선 가능: API는 업데이트되고 개선되어야 하며, 사용자에게 더 좋은 서비스를 제공하려면 이 점을 고려해

❓ 질문: 파이썬에서 비동기 프로그래밍의 장점은?

🟢 SFT+DPO 응답:
   비동기 프로그래밍은 파이썬에서도 많은 이점을 제공합니다. 몇 가지 주요 장점으로는 다음과 같습니다

---
## 7️⃣ 선호도 정렬 평가

DPO 학습이 실제로 선호도 정렬에 효과가 있는지 정량적으로 평가합니다.

In [17]:
# 선호도 정렬 정성적 평가
print("=" * 60)
print("📌 선호도 정렬 평가")
print("=" * 60)

eval_criteria = [
    "응답의 구조성 (번호, 소제목 사용)",
    "응답의 상세함 (구체적 예시 포함)",
    "응답의 친절함 (교육적 톤)",
    "응답의 정확성 (사실에 기반)",
]

print("\n📋 DPO 학습 후 기대되는 변화:")
for i, criterion in enumerate(eval_criteria, 1):
    print(f"   {i}. {criterion}")
    print(f"      SFT Only: 기본적인 답변 가능")
    print(f"      SFT+DPO: chosen 스타일에 더 가깝게 답변")

print("\n💡 DPO의 효과:")
print("   • 모델이 'chosen' 스타일(구조적, 상세한, 친절한)로 답변하도록 유도")
print("   • 'rejected' 스타일(짧고 불친절한)의 답변 확률 감소")
print("   • 데이터 양이 많을수록 효과가 더 뚜렷해짐")

📌 선호도 정렬 평가

📋 DPO 학습 후 기대되는 변화:
   1. 응답의 구조성 (번호, 소제목 사용)
      SFT Only: 기본적인 답변 가능
      SFT+DPO: chosen 스타일에 더 가깝게 답변
   2. 응답의 상세함 (구체적 예시 포함)
      SFT Only: 기본적인 답변 가능
      SFT+DPO: chosen 스타일에 더 가깝게 답변
   3. 응답의 친절함 (교육적 톤)
      SFT Only: 기본적인 답변 가능
      SFT+DPO: chosen 스타일에 더 가깝게 답변
   4. 응답의 정확성 (사실에 기반)
      SFT Only: 기본적인 답변 가능
      SFT+DPO: chosen 스타일에 더 가깝게 답변

💡 DPO의 효과:
   • 모델이 'chosen' 스타일(구조적, 상세한, 친절한)로 답변하도록 유도
   • 'rejected' 스타일(짧고 불친절한)의 답변 확률 감소
   • 데이터 양이 많을수록 효과가 더 뚜렷해짐


In [18]:
# DPO 학습 로그 분석
print("=" * 60)
print("📌 DPO 학습 로그 분석")
print("=" * 60)

# 학습 이력에서 주요 메트릭 추출
if hasattr(dpo_trainer, 'state') and dpo_trainer.state.log_history:
    log_history = dpo_trainer.state.log_history
    
    print("\n📊 학습 이력:")
    for entry in log_history:
        if 'loss' in entry:
            step = entry.get('step', '?')
            loss = entry.get('loss', '?')
            lr = entry.get('learning_rate', '?')
            print(f"   Step {step}: loss={loss:.4f}" if isinstance(loss, float) else f"   Step {step}: loss={loss}")
    
    # DPO 특유의 메트릭
    print("\n📊 DPO 메트릭 설명:")
    print("   • loss: DPO 손실 (낮을수록 좋음)")
    print("   • rewards/chosen: chosen 응답에 대한 암시적 보상")
    print("   • rewards/rejected: rejected 응답에 대한 암시적 보상")
    print("   • rewards/margins: chosen - rejected (양수가 좋음)")
else:
    print("\n⚠️ 학습 로그를 찾을 수 없습니다.")

📌 DPO 학습 로그 분석

📊 학습 이력:
   Step 1: loss=0.4006
   Step 2: loss=0.4584
   Step 3: loss=0.3124
   Step 4: loss=0.1972
   Step 5: loss=0.1763
   Step 6: loss=0.1346

📊 DPO 메트릭 설명:
   • loss: DPO 손실 (낮을수록 좋음)
   • rewards/chosen: chosen 응답에 대한 암시적 보상
   • rewards/rejected: rejected 응답에 대한 암시적 보상
   • rewards/margins: chosen - rejected (양수가 좋음)


In [19]:
# GPU 메모리 정리
print("🧹 GPU 메모리 최종 정리...")
print_gpu_memory("정리 전")

del dpo_trainer
del dpo_model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료")

🧹 GPU 메모리 최종 정리...
[정리 전] GPU: 1.5GB / 23.5GB
[정리 후] GPU: 0.9GB / 23.5GB
✅ 메모리 정리 완료


---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **SFT → DPO 파이프라인**: 2단계로 모델을 점진적으로 개선

2️⃣ **4bit + LoRA**: RTX 4060(8GB)에서 DPO 학습 가능

3️⃣ **DPO 핵심 파라미터**:
   - `beta=0.1`: KL 페널티 강도 (선호도 학습 강도 조절)
   - `loss_type='sigmoid'`: 기본 DPO 손실 함수

4️⃣ **DPO의 효과**:
   - Chosen 스타일(구조적, 상세, 친절)로 답변 유도
   - Rejected 스타일(짧고 불친절) 답변 확률 감소

5️⃣ **RTX 4060 설정**:
   - `per_device_train_batch_size=1`
   - `gradient_accumulation_steps=8`
   - `fp16=True`
   - `max_seq_length=1024`

### 생성된 파일

| 파일 | 설명 |
|------|------|
| `outputs/dpo_training/sft_checkpoint/` | SFT 모델 체크포인트 |
| `outputs/dpo_training/dpo_checkpoint/` | DPO 모델 체크포인트 |

### 다음 세션 예고

- 📘 **Session 29**: DeepSeek R1 Case Study with GRPO
  - GRPO 알고리즘으로 수학 추론 능력 향상

In [20]:
# 최종 확인
print("=" * 60)
print("📌 최종 확인")
print("=" * 60)

# 저장된 체크포인트 확인
for checkpoint_dir in [SFT_OUTPUT, DPO_OUTPUT]:
    if checkpoint_dir.exists():
        files = list(checkpoint_dir.glob("*"))
        total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
        print(f"  ✅ {checkpoint_dir.name}: {len(files)} 파일, {total_size:.1f}MB")
    else:
        print(f"  ⚠️ {checkpoint_dir.name}: 디렉토리 없음")

print("\n" + "=" * 60)
print("✅ Session 27 완료!")
print("   SFT → DPO 파이프라인을 성공적으로 실습했습니다.")
print("   다음 세션에서 GRPO를 활용한 추론 모델 학습을 진행합니다.")

📌 최종 확인
  ✅ sft_checkpoint: 21 파일, 31.8MB
  ✅ dpo_checkpoint: 14 파일, 31.8MB

✅ Session 27 완료!
   SFT → DPO 파이프라인을 성공적으로 실습했습니다.
   다음 세션에서 GRPO를 활용한 추론 모델 학습을 진행합니다.
